In [ ]:
# %%
import pandas as pd
from pathlib import Path

# Raiz do projeto: a pasta que contém UCMF_raw.csv. Procurar a partir da pasta atual
# para cima faz o notebook funcionar rodando da própria subpasta ou da raiz.
RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "UCMF_raw.csv").exists())
PASTA = RAIZ / "Análise Univariada"   # tabelas e figuras desta análise
DOCS = RAIZ / "Docs"                  # documentação (regra de conversão)

df = pd.read_csv(RAIZ / "UCMF_raw.csv", dtype=str, keep_default_na=False)

print(df.shape)
print(df.columns.tolist())

In [ ]:
# %%
def converte_data(coluna):
    texto = pd.to_datetime(coluna, format="%d/%m/%y", errors="coerce")              #tenta ler a string no formato dia/mês/ano-2-dígitos. errors="coerce" faz o que não bate virar NaT (o "vazio" do tipo data) em vez de travar o script
    serial = pd.to_numeric(coluna, errors="coerce")                         #tenta ler como número puro; texto vira NaN
    excel = pd.to_datetime(serial, unit="D", origin="1899-12-30", errors="coerce")      #interpreta esse número como "dias desde 30/12/1899" — é a contagem interna que o Excel usa para guardar datas.
    return texto.fillna(excel)

df["Atendimento_dt"] = converte_data(df["Atendimento"])
df["DN_dt"] = converte_data(df["DN"])

for original, nova in [("Atendimento", "Atendimento_dt"), ("DN", "DN_dt")]:
    vazio = (df[original].str.strip() == "").sum()
    invalido = ((df[original].str.strip() != "") & df[nova].isna()).sum()
    valido = df[nova].notna().sum()
    print(f"{original}: válido={valido} vazio={vazio} formato_inválido={invalido}")
    print(f"  min={df[nova].min()}  max={df[nova].max()}")

In [ ]:
# %%
df["FC_num"] = pd.to_numeric(df["FC"], errors="coerce")

vazio = (df["FC"].str.strip() == "").sum()
invalido_mask = df["FC"].str.strip().ne("") & df["FC_num"].isna()
invalido = invalido_mask.sum()
valido = df["FC_num"].notna().sum()

print(f"FC: válido={valido} vazio={vazio} texto_não_numérico={invalido}")
print("exemplos de texto não numérico:", sorted(df["FC"][invalido_mask].unique())[:5])
print(df["FC_num"].describe())

In [ ]:
# %%
for col in ["Peso", "Altura", "IMC", "IDADE"]:
    df[col + "_num"] = pd.to_numeric(df[col], errors="coerce")

print("Peso   -> zeros:", (df["Peso_num"]==0).sum(), " | negativos:", (df["Peso_num"]<0).sum())
print("Altura -> zeros:", (df["Altura_num"]==0).sum(), " | valor 99:", (df["Altura_num"]==99).sum())
print("IMC    -> zeros:", (df["IMC_num"]==0).sum())
print("IDADE  -> zeros:", (df["IDADE_num"]==0).sum(), " | negativos:", (df["IDADE_num"]<0).sum(), " | mínimo:", df["IDADE_num"].min())

# aplica a regra mínima: só o que é fisicamente impossível vira ausente
df["Peso_limpo"]   = df["Peso_num"].mask(df["Peso_num"] == 0)
df["Altura_limpo"] = df["Altura_num"].mask(df["Altura_num"] == 0)   # 99 fica — ver explicação
df["IMC_limpo"]    = df["IMC_num"].mask(df["IMC_num"] == 0)
df["IDADE_limpo"]  = df["IDADE_num"]                                 # nada convertido aqui

print("\nn válido após a conversão:")
for col in ["Peso_limpo", "Altura_limpo", "IMC_limpo", "IDADE_limpo"]:
    print(f"  {col}: {df[col].notna().sum()}")

In [ ]:
# %%
print(df["PPA"].value_counts(dropna=False))

codigos_faltantes_ppa = ["", "Não Calculado", "#VALUE!"]
df["PPA_limpo"] = df["PPA"].replace(codigos_faltantes_ppa, pd.NA)

print("\nn válido PPA:", df["PPA_limpo"].notna().sum())
print("n ausente (vazio + código):", df["PPA_limpo"].isna().sum())

In [ ]:
# %%
cols_categoricas = ["PULSOS","B2","SOPRO","NORMAL X ANORMAL","SEXO","HDA 1","HDA2","MOTIVO1","MOTIVO2"]
for col in cols_categoricas:
    df[col] = df[col].str.strip()

# mapeamentos explícitos: só grafia, nunca junta conceito clínico diferente
mapa_pulsos = {"NORMAIS": "Normais", "AMPLOS": "Amplos"}
mapa_sopro  = {"sistólico": "Sistólico", "contínuo": "Contínuo"}
mapa_alvo   = {"anormal": "Anormal", "Normais": "Normal"}
mapa_sexo   = {"M": "Masculino", "masculino": "Masculino", "F": "Feminino"}

df["PULSOS_pad"] = df["PULSOS"].replace(mapa_pulsos)
df["SOPRO_pad"]  = df["SOPRO"].replace(mapa_sopro)
df["ALVO_pad"]   = df["NORMAL X ANORMAL"].replace(mapa_alvo)
df["SEXO_pad"]   = df["SEXO"].replace(mapa_sexo)

for original, pad in [("PULSOS","PULSOS_pad"), ("SOPRO","SOPRO_pad"),
                       ("NORMAL X ANORMAL","ALVO_pad"), ("SEXO","SEXO_pad")]:
    brutas = df[original][df[original]!=""].nunique()
    padronizadas = df[pad][df[pad]!=""].nunique()
    print(f"{original}: {brutas} brutas -> {padronizadas} padronizadas")

df["SEXO_pad"] = df["SEXO_pad"].replace("Indeterminado", "")

In [ ]:
# %%
regra_conversao = pd.DataFrame([
    ["Atendimento", "dd/mm/aa",                     "formato recuperável",   "→ data",                        11578],
    ["Atendimento", "número de 5 dígitos",           "serial do Excel",       "→ data (origem 30/12/1899)",     312],
    ["DN",          "dd/mm/aa",                      "formato recuperável",   "→ data",                        11253],
    ["DN",          "número de 5 dígitos",           "serial do Excel",       "→ data (origem 30/12/1899)",     124],
    ["DN",          'ex.: "4/anos", "18/10,1996"',   "formato inválido",      "→ ausente",                       120],
    ["FC",          'ex.: "80-90", "120-160"',       "não numérico",          "→ ausente",                        16],
    ["Peso",        "0",                             "código de faltante",    "→ ausente",                      2609],
    ["Altura",      "0",                             "código de faltante",    "→ ausente",                      4460],
    ["IMC",         "0",                             "código de faltante",    "→ ausente",                       109],
    ["PPA",         "Não Calculado",                 "código de faltante",    "→ ausente",                      9081],
    ["PPA",         "#VALUE!",                       "código de faltante",    "→ ausente",                      1496],
    ["PULSOS",      "NORMAIS, AMPLOS",               "grafia inconsistente",  "→ forma padrão",                    3],
    ["SOPRO",       "sistólico, contínuo",           "grafia inconsistente",  "→ forma padrão",                  751],
    ["NORMAL X ANORMAL (alvo)", "anormal, Normais",  "grafia inconsistente",  "→ forma padrão",                    2],
    ["SEXO",        "M, F, masculino",               "grafia inconsistente",  "→ forma padrão",                10621],
    ["SEXO", "Indeterminado", "código de faltante", "→ ausente", 1417],
], columns=["Variável","Valor encontrado","Tipo","Ação","n afetados"])

print(regra_conversao.to_string(index=False))
regra_conversao.to_csv(DOCS / "regra_conversao.csv", index=False)

In [ ]:
# %%
df["PAS_num"] = pd.to_numeric(df["PA SISTOLICA"], errors="coerce")
df["PAD_num"] = pd.to_numeric(df["PA DIASTOLICA"], errors="coerce")

variaveis_A = {
    "Peso": "Peso_limpo", "Altura": "Altura_limpo", "IMC": "IMC_limpo",
    "IDADE": "IDADE_limpo", "PAS": "PAS_num", "PAD": "PAD_num", "FC": "FC_num",
}

n_total = len(df)
linhas = []
for nome, col in variaveis_A.items():
    s = df[col]
    desc = s.describe()
    linhas.append({
        "Variável": nome,
        "n válido": int(desc["count"]),
        "% ausente": round((n_total - desc["count"]) / n_total * 100, 1),
        "mín": round(desc["min"], 2), "Q1": round(desc["25%"], 2),
        "mediana": round(desc["50%"], 2), "média": round(desc["mean"], 2),
        "Q3": round(desc["75%"], 2), "máx": round(desc["max"], 2),
        "DP": round(desc["std"], 2), "IQR": round(desc["75%"] - desc["25%"], 2),
        "assimetria": round(s.skew(), 2),
    })

tabela_A = pd.DataFrame(linhas)
print(tabela_A.to_string(index=False))
tabela_A.to_csv(PASTA / "tabela_A.csv", index=False)

In [ ]:
# %%
variaveis_B = {
    "PULSOS": ("PULSOS", "PULSOS_pad"),
    "B2": ("B2", "B2"),
    "SOPRO": ("SOPRO", "SOPRO_pad"),
    "PPA": ("PPA", "PPA_limpo"),
    "NORMAL X ANORMAL (alvo)": ("NORMAL X ANORMAL", "ALVO_pad"),
    "SEXO": ("SEXO", "SEXO_pad"),
    "HDA 1": ("HDA 1", "HDA 1"),
    "HDA2": ("HDA2", "HDA2"),
    "MOTIVO1": ("MOTIVO1", "MOTIVO1"),
    "MOTIVO2": ("MOTIVO2", "MOTIVO2"),
}

linhas = []
for nome, (bruta, pad) in variaveis_B.items():
    s = df[pad].replace("", pd.NA)
    validos = s.dropna()
    contagem = validos.value_counts()
    n_moda = contagem.max()
    modas = contagem[contagem == n_moda].index.tolist()
    linhas.append({
        "Variável": nome,
        "n válido": len(validos),
        "% ausente": round(s.isna().mean() * 100, 1),
        "categ. brutas": df[bruta][df[bruta].str.strip() != ""].nunique(),
        "categ. padronizadas": validos.nunique(),
        "moda": " / ".join(modas),
        "% moda (÷ n total)": round(n_moda / n_total * 100, 1),
        "% moda (÷ n válido)": round(n_moda / len(validos) * 100, 1),
    })

tabela_B = pd.DataFrame(linhas)
print(tabela_B.to_string(index=False))
tabela_B.to_csv(PASTA / "tabela_B.csv", index=False)

In [ ]:
# %%
hoje = pd.Timestamp("2026-09-25")

# ID
print(f"ID: {df['ID'].nunique()} distintos em {n_total} registros")

# Convenio
conv = df["Convenio"][df["Convenio"].str.strip() != ""]
print(f"Convenio: n válido {len(conv)}; % ausente {round((n_total - len(conv)) / n_total * 100, 1)}; "
      f"{conv.nunique()} grafias distintas ({conv.str.strip().str.upper().nunique()} se ignorar maiúsc./minúsc.)")

# DN
dn_vazio = (df["DN"].str.strip() == "").sum()
dn_invalido = ((df["DN"].str.strip() != "") & df["DN_dt"].isna()).sum()
print(f"DN: n válido {df['DN_dt'].notna().sum()} (vazios {dn_vazio}; formato inválido {dn_invalido}); "
      f"de {df['DN_dt'].min().date()} a {df['DN_dt'].max().date()}; datas no futuro: {(df['DN_dt'] > hoje).sum()}")

# Atendimento
print(f"Atendimento: n válido {df['Atendimento_dt'].notna().sum()}; "
      f"de {df['Atendimento_dt'].min().date()} a {df['Atendimento_dt'].max().date()}; "
      f"datas no futuro: {(df['Atendimento_dt'] > hoje).sum()}")

In [ ]:
# %%
# Variáveis que não entram nas tabelas A e B (identificador, administrativa e datas): uma
# linha para cada, com n válido, % ausente e o problema principal, para que a análise
# univariada cubra todas as variáveis da base.
serial_dn = df["DN"].str.strip().str.fullmatch(r"[0-9]{5}")
serial_atend = df["Atendimento"].str.strip().str.fullmatch(r"[0-9]{5}")


def pct_ausente(n_validos):
    return round((n_total - n_validos) / n_total * 100, 1)


n_id = int((df["ID"].str.strip() != "").sum())
n_dn = int(df["DN_dt"].notna().sum())
n_atend = int(df["Atendimento_dt"].notna().sum())

tabela_C = pd.DataFrame([
    {"Variável": "ID", "n válido": n_id, "% ausente": pct_ausente(n_id),
     "problema principal": f"identificador: {df['ID'].nunique()} valores distintos em {n_total} "
                           "registros; sem informação clínica"},
    {"Variável": "Convenio", "n válido": len(conv), "% ausente": pct_ausente(len(conv)),
     "problema principal": f"{conv.nunique()} grafias distintas ({conv.str.strip().str.upper().nunique()} "
                           "ignorando maiúsc./minúsc.) para os mesmos planos; sem relação clínica com o alvo"},
    {"Variável": "DN", "n válido": n_dn, "% ausente": pct_ausente(n_dn),
     "problema principal": f"{dn_invalido} valores em formato inválido (ex.: '4/anos'), "
                           f"{int(serial_dn.sum())} datas gravadas como serial do Excel e "
                           f"{int((df['DN_dt'] > hoje).sum())} datas no futuro (ano de 2 dígitos lido "
                           "no século errado)"},
    {"Variável": "Atendimento", "n válido": n_atend, "% ausente": pct_ausente(n_atend),
     "problema principal": f"{int(serial_atend.sum())} datas gravadas como serial do Excel e "
                           f"{int((df['Atendimento_dt'] > hoje).sum())} data(s) no futuro"},
])
print(tabela_C.to_string(index=False))
tabela_C.to_csv(PASTA / "tabela_C.csv", index=False)

In [ ]:
# %%
import matplotlib.pyplot as plt

AZUL, TEXTO, TEXTO2, GRADE = "#2a78d6", "#0b0b0b", "#52514e", "#e5e5e2"
N_FAIXAS = 30
unidades = {"Peso": "kg", "Altura": "cm", "IMC": "kg/m²", "IDADE": "anos",
            "PAS": "mmHg", "PAD": "mmHg", "FC": "bpm"}

fig, eixos = plt.subplots(2, 4, figsize=(14, 6.5))
eixos = eixos.flatten()

for ax, (nome, col) in zip(eixos, variaveis_A.items()):
    s = df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lim_inf = max(s.min(), q1 - 5 * iqr)
    lim_sup = min(s.max(), q3 + 5 * iqr)
    dentro = s[(s >= lim_inf) & (s <= lim_sup)]
    fora = len(s) - len(dentro)

    ax.hist(dentro, bins=N_FAIXAS, color=AZUL, edgecolor="white", linewidth=0.8)
    ax.set_title(f"{nome} (n = {len(s)})", loc="left", fontsize=11, color=TEXTO)
    ax.set_xlabel(unidades[nome], fontsize=9, color=TEXTO2)
    ax.text(0.98, 0.95, f"{fora} valor(es) fora do eixo\nmín {s.min():g} | máx {s.max():g}",
            transform=ax.transAxes, ha="right", va="top", fontsize=8, color=TEXTO2)
    ax.grid(axis="y", color=GRADE, linewidth=0.6)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(colors=TEXTO2, labelsize=8)

eixos[-1].axis("off")
eixos[0].set_ylabel("frequência", fontsize=9, color=TEXTO2)
eixos[4].set_ylabel("frequência", fontsize=9, color=TEXTO2)
fig.suptitle("Figura 1 – Histogramas das variáveis quantitativas após a conversão mínima "
             "(30 faixas; eixo x limitado a Q1 − 5·IQR e Q3 + 5·IQR)",
             x=0.01, ha="left", fontsize=11, color=TEXTO)
fig.tight_layout()
fig.savefig(PASTA / "fig1_histogramas.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# %%
fig, eixos = plt.subplots(7, 1, figsize=(10, 9))

for ax, (nome, col) in zip(eixos, variaveis_A.items()):
    s = df[col].dropna()
    ax.boxplot(s, orientation="horizontal", whis=(10, 90), widths=0.6, patch_artist=True,
               boxprops=dict(facecolor=AZUL, edgecolor=AZUL, alpha=0.35),
               medianprops=dict(color=AZUL, linewidth=2),
               whiskerprops=dict(color=TEXTO2), capprops=dict(color=TEXTO2),
               flierprops=dict(marker="o", markersize=3, markerfacecolor=TEXTO2,
                               markeredgecolor="none", alpha=0.3))
    ax.set_yticks([1])
    ax.set_yticklabels([f"{nome} ({unidades[nome]})"], fontsize=9, color=TEXTO)
    ax.grid(axis="x", color=GRADE, linewidth=0.6)
    ax.set_axisbelow(True)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="x", colors=TEXTO2, labelsize=8)
    ax.tick_params(axis="y", length=0)

fig.suptitle("Figura 2 – Box plots das variáveis quantitativas após a conversão mínima "
             "(caixa: Q1–Q3; linha: mediana; bigodes: P10 e P90; pontos: valores fora de P10–P90)",
             x=0.01, ha="left", fontsize=10, color=TEXTO)
fig.tight_layout()
fig.savefig(PASTA / "fig2_boxplots.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# %%
LARANJA = "#eb6834"

colunas_finais = {
    "Peso": ("Peso", "Peso_limpo"), "Altura": ("Altura", "Altura_limpo"),
    "IMC": ("IMC", "IMC_limpo"), "IDADE": ("IDADE", "IDADE_limpo"),
    "PAS": ("PA SISTOLICA", "PAS_num"), "PAD": ("PA DIASTOLICA", "PAD_num"),
    "FC": ("FC", "FC_num"), "Atendimento": ("Atendimento", "Atendimento_dt"),
    "DN": ("DN", "DN_dt"), "Convenio": ("Convenio", "Convenio"),
    "PULSOS": ("PULSOS", "PULSOS_pad"), "B2": ("B2", "B2"), "SOPRO": ("SOPRO", "SOPRO_pad"),
    "PPA": ("PPA", "PPA_limpo"), "NORMAL X ANORMAL": ("NORMAL X ANORMAL", "ALVO_pad"),
    "SEXO": ("SEXO", "SEXO_pad"), "HDA 1": ("HDA 1", "HDA 1"), "HDA2": ("HDA2", "HDA2"),
    "MOTIVO1": ("MOTIVO1", "MOTIVO1"), "MOTIVO2": ("MOTIVO2", "MOTIVO2"),
}

linhas = []
for nome, (bruta, final) in colunas_finais.items():
    vazio = (df[bruta].str.strip() == "").sum()
    ausente_final = df[final].replace("", pd.NA).isna().sum()
    linhas.append({"Variável": nome,
                   "vazio": vazio / n_total * 100,
                   "código/inválido": (ausente_final - vazio) / n_total * 100})

ausentes = pd.DataFrame(linhas).set_index("Variável")
ausentes["total"] = ausentes["vazio"] + ausentes["código/inválido"]
ausentes = ausentes.sort_values("total")
print(ausentes.round(1))

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(ausentes.index, ausentes["vazio"], color=AZUL, edgecolor="white", linewidth=2, label="vazio")
ax.barh(ausentes.index, ausentes["código/inválido"], left=ausentes["vazio"], color=LARANJA,
        edgecolor="white", linewidth=2, label="código de faltante / inválido")
for i, total in enumerate(ausentes["total"]):
    rotulo = f"{total:.1f}%" if total >= 0.1 else "<0.1%"
    ax.text(total + 1, i, rotulo, va="center", fontsize=8, color=TEXTO2)

ax.set_xlim(0, 105)
ax.set_xlabel("% dos 12873 registros", fontsize=9, color=TEXTO2)
ax.grid(axis="x", color=GRADE, linewidth=0.6)
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="x", colors=TEXTO2, labelsize=8)
ax.tick_params(axis="y", colors=TEXTO, labelsize=9, length=0)
ax.legend(loc="lower right", frameon=False, fontsize=9)
ax.set_title("Figura 3 – Proporção de ausentes por variável após a conversão mínima",
             loc="left", fontsize=11, color=TEXTO)
fig.tight_layout()
fig.savefig(PASTA / "fig3_ausentes.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# %%
ausente = pd.DataFrame({nome: df[final].replace("", pd.NA).isna()
                        for nome, (bruta, final) in colunas_finais.items()})
estrutural_HDA2 = (df["HDA2"] == "") & (df["HDA 1"] != "")
estrutural_MOTIVO2 = (df["MOTIVO2"] == "") & (df["MOTIVO1"] == "2 - Check-up")
ausente.loc[estrutural_HDA2, "HDA2"] = False
ausente.loc[estrutural_MOTIVO2, "MOTIVO2"] = False
frac_ausente = ausente.mean(axis=1)
atributos_brutos = [bruta for bruta, final in colunas_finais.values()]

passo1 = df.duplicated(subset=atributos_brutos, keep="first")
passo2 = ~passo1 & (frac_ausente > 0.5)
passo3 = ~passo1 & ~passo2 & ausente["NORMAL X ANORMAL"]
manter = ~(passo1 | passo2 | passo3)

print("1) duplicatas exatas (sem ID):", passo1.sum())
print("2) mais de 50% dos atributos ausentes:", passo2.sum())
print("3) sem alvo:", passo3.sum())
print("restam:", manter.sum(), "de", n_total)

for limiar in [0.4, 0.5, 0.6]:
    print(f"   limiar {limiar:.0%}: {(~passo1 & (frac_ausente > limiar)).sum()} objetos")

antes = df["ALVO_pad"].replace("", pd.NA).dropna()
depois = df.loc[manter, "ALVO_pad"]
print(f"% Anormal: antes {(antes == 'Anormal').mean()*100:.1f} | depois {(depois == 'Anormal').mean()*100:.1f}")

df_limpo = df[manter].copy()